# do-Shapley Quick Validation (Tabular Data)

Quick validation of the `DoCausalImputer` bug fixes on the Bike Sharing dataset (11 features, 587 train samples).  
Tests Marginal SHAP / DML / IPW / REG on 3 test samples. **Runtime: ~50 minutes.**

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import time as time_module

import shapiq
from shapiq import TabularExplainer
from shapiq.imputer import DoCausalImputer
from shapiq.datasets import load_bike_sharing_daily, load_bike_sharing_daily_train_index

print(f"shapiq version: {shapiq.__version__}")

e:\Desktop\Shapley\Shapiq\shapiq\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


shapiq version: None


In [2]:
# Load data
x_preprocessed, y_data = load_bike_sharing_daily(preprocessed=True)
x_raw, _ = load_bike_sharing_daily(preprocessed=False)
train_idx = load_bike_sharing_daily_train_index()

x_data_raw = x_preprocessed.copy()
x_data_raw['weekday'] = x_raw['weekday'].values
x_data_raw['workingday'] = x_raw['workingday'].values
x_data_raw['holiday'] = x_raw['holiday'].values
x_data_raw['weather'] = x_raw['weathersit'].values
x_data_raw['location'] = 1.0
x_data_raw.rename(columns={'hum': 'humidity', 'atemp': 'feel_temp'}, inplace=True)

feature_names = [
    "trend", "sinyear", "cosyear", "weekday", "location", "workingday",
    "holiday", "weather", "temp", "humidity", "windspeed", "feel_temp",
]
x_data = x_data_raw[feature_names].copy()

all_idx = np.arange(len(x_data))
test_idx = np.array([i for i in all_idx if i not in train_idx])

X_train = x_data.iloc[train_idx].values
y_train_nc = y_data.iloc[train_idx].values
y_train_mean = y_train_nc.mean()
y_train = y_train_nc - y_train_mean

X_test = x_data.iloc[test_idx].values
y_test = y_data.iloc[test_idx].values - y_train_mean
n_features = len(feature_names)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (587, 12), Test: (144, 12)


In [3]:
# Train model
model = xgb.XGBRegressor(n_estimators=100, random_state=1, verbosity=0)
model.fit(X_train, y_train)
rmse = np.sqrt(np.mean((y_test - model.predict(X_test)) ** 2))
print(f"Test RMSE: {rmse:.2f}")

Test RMSE: 721.46


In [4]:
# DAG
# 0:trend 1:sinyear 2:cosyear 3:weekday 4:location
# 5:workingday 6:holiday 7:weather 8:temp 9:humidity 10:windspeed 11:feel_temp
dag_edges = [
    (0, 5), (0, 6), (3, 5), (3, 6), (4, 5), (4, 6),   # trend/weekday/location → workingday, holiday
    (0, 1), (0, 2), (0, 7), (0, 8), (0, 9), (0, 10),   # trend → sinyear, cosyear, weather, temp, humidity, windspeed
    (4, 7), (4, 8), (4, 9), (4, 10),                     # location → weather, temp, humidity, windspeed
    (7, 11), (8, 11), (9, 11), (10, 11),                 # weather, temp, humidity, windspeed → feel_temp
]
confounding_pairs = DoCausalImputer.confounding_group(7, 8, 9, 10)

In [5]:
# Create explainers
N_QUICK = 3
budget = 2048
RANDOM_STATE = 42

explainers = {}
explainers['Marginal SHAP'] = TabularExplainer(
    model=model.predict, data=X_train, imputer="marginal",
    index="SV", max_order=1, random_state=RANDOM_STATE
)
for method in ['dml', 'ipw', 'reg']:
    explainers[f'do-Shapley ({method.upper()})'] = TabularExplainer(
        model=model.predict, data=X_train, imputer="do_causal",
        index="SV", max_order=1, random_state=RANDOM_STATE,
        dag_edges=dag_edges, confounding_pairs=confounding_pairs, method=method,
    )

# Verify conditioning sets
imp = explainers['do-Shapley (DML)'].imputer
print(f"cross_fitting: {imp.cross_fitting}")
for vi in imp.topo_order:
    mi = imp._cond_prob_models[vi]
    if mi['type'] == 'marginal':
        print(f"  {feature_names[vi]}: marginal")
    else:
        print(f"  {feature_names[vi]}: P(· | {[feature_names[p] for p in mi['pre_indices']]})")

cross_fitting: True
  location: marginal
  weekday: P(· | ['location'])
  trend: P(· | ['location', 'weekday'])
  windspeed: P(· | ['location', 'weekday', 'trend', 'humidity', 'weather', 'temp'])
  humidity: P(· | ['location', 'weekday', 'trend', 'windspeed', 'weather', 'temp'])
  temp: P(· | ['location', 'weekday', 'trend', 'windspeed', 'humidity', 'weather'])
  weather: P(· | ['location', 'weekday', 'trend', 'windspeed', 'humidity', 'temp'])
  feel_temp: P(· | ['location', 'weekday', 'trend', 'windspeed', 'humidity', 'temp', 'weather'])
  cosyear: P(· | ['location', 'weekday', 'trend', 'windspeed', 'humidity', 'temp', 'weather', 'feel_temp'])
  sinyear: P(· | ['location', 'weekday', 'trend', 'windspeed', 'humidity', 'temp', 'weather', 'feel_temp', 'cosyear'])
  holiday: P(· | ['location', 'weekday', 'trend', 'windspeed', 'humidity', 'temp', 'weather', 'feel_temp', 'cosyear', 'sinyear'])
  workingday: P(· | ['location', 'weekday', 'trend', 'windspeed', 'humidity', 'temp', 'weather', '

In [6]:
# Compute Shapley values (~50 min total)
sv = {name: np.zeros((N_QUICK, n_features)) for name in explainers}

t_total = time_module.time()
for name, exp in explainers.items():
    t0 = time_module.time()
    for i in range(N_QUICK):
        result = exp.explain(X_test[i:i+1], budget=budget)
        sv[name][i] = result.get_n_order_values(order=1)
    print(f"  {name}: {time_module.time()-t0:.1f}s")
print(f"Total: {time_module.time()-t_total:.1f}s")

  Marginal SHAP: 1.9s
  do-Shapley (DML): 3105.2s
  do-Shapley (IPW): 72.0s
  do-Shapley (REG): 680.4s
Total: 3859.5s


In [7]:
# Results
print(f"{'Feature':>12s} {'Marginal':>10s} {'DML':>10s} {'IPW':>10s} {'REG':>10s}")
print(f"{'-'*12} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
for f_idx, fname in enumerate(feature_names):
    vals = [np.abs(sv[n][:, f_idx]).mean() for n in explainers]
    print(f"{fname:>12s} {vals[0]:>10.1f} {vals[1]:>10.1f} {vals[2]:>10.1f} {vals[3]:>10.1f}")

# location diagnostic
loc_idx = feature_names.index('location')
print(f"\nlocation (const=1) importance:")
for name in explainers:
    v = np.abs(sv[name][:, loc_idx]).mean()
    print(f"  {name}: {v:.1f}  {'✗' if v > 200 else '✓'}")

# temp/sinyear ratio
temp_idx, sin_idx = feature_names.index('temp'), feature_names.index('sinyear')
print(f"\ntemp/sinyear ratio:")
for name in explainers:
    t = np.abs(sv[name][:, temp_idx]).mean()
    s = np.abs(sv[name][:, sin_idx]).mean()
    print(f"  {name}: {t/s:.2f}")

     Feature   Marginal        DML        IPW        REG
------------ ---------- ---------- ---------- ----------
       trend     1773.7     2058.5      280.0     1569.4
     sinyear       65.0       27.6      120.4       81.9
     cosyear      110.1       51.6      250.6      159.8
     weekday       71.8       63.7      254.2       63.3
    location       16.7       35.0      248.0       13.1
  workingday       64.3       72.9      144.7       57.7
     holiday       18.1       11.1      148.9        6.8
     weather       79.6       73.2      175.5       63.9
        temp      980.0      693.2      496.9      712.0
    humidity      207.6       75.2      265.1      118.3
   windspeed      125.0       51.0      217.9       85.0
   feel_temp      492.9      231.1      499.2      670.8

location (const=1) importance:
  Marginal SHAP: 16.7  ✓
  do-Shapley (DML): 35.0  ✓
  do-Shapley (IPW): 248.0  ✗
  do-Shapley (REG): 13.1  ✓

temp/sinyear ratio:
  Marginal SHAP: 15.08
  do-Shapley (DM